# Baseline Comparisons — TE Classification

This notebook trains and evaluates baseline models for comparison against the advanced **v4.3 Hybrid (CNN+GNN)** model. It produces the **baseline comparison bar chart** for the thesis Discussion section (§4.3 — Comparative Performance).

## Models evaluated

| Model | Type | Description |
|-------|------|-------------|
| Logistic Regression | ML | k-mer frequency features (mean-pooled windows) |
| Random Forest | ML | k-mer frequency features (mean-pooled windows) |
| Standard CNN | CNN | Motif convolutions + global pool, **no** dilated context blocks |
| Dilated CNN | CNN | Motif convolutions + dilated context blocks, **no** GNN |
| v4.3 Hybrid | CNN + GNN | Full model — loaded from saved test predictions |

**Data split**: identical to v4.3 (`test_size=0.2`, `random_state=42`, same `EXCLUDE_GENOMES`)  
**Output figure**: `../../thesis/figures/baseline_comparison.png`

In [2]:
# ============ Imports ============
import gc
import time
import json as _json
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.9.0


In [3]:
# ============ Configuration ============
# Paths are relative to this notebook: data_analysis/baseline_comparisons/
FASTA_PATH     = "../../data/vgp/all_vgp_tes.fa"
LABEL_PATH     = "../../data/vgp/20260120_features_sf"
RESULTS_V43    = "../vgp_model_data_tpase_multi/v4.3/results_v4.3.pt"
FIGURE_PATH    = "../../thesis/figures/baseline_comparison.png"
CHECKPOINT_DIR = Path(".")   # baseline_comparisons/ — checkpoints saved here

# Data — identical to v4.3 for a fair comparison
EXCLUDE_GENOMES  = {'mOrnAna', 'bTaeGut', 'rAllMis'}
KEEP_CLASSES     = ['DNA', 'LTR', 'LINE']
MIN_CLASS_COUNT  = 100
MAX_PER_SF       = 3000
TEST_SIZE        = 0.2
RANDOM_STATE     = 42

# K-mer parameters — identical to v4.3
KMER_K      = 7
KMER_DIM    = 2048
KMER_WINDOW = 512
KMER_STRIDE = 256

# CNN baseline hyperparameters
FIXED_LENGTH      = 20000
CNN_WIDTH         = 128
MOTIF_KERNELS     = (7, 15, 21)
CONTEXT_DILATIONS = (1, 2, 4, 8)
DROPOUT           = 0.15
BASELINE_EPOCHS   = 20        # fewer than v4.3's 40; sufficient for baselines
BASELINE_LR       = 1e-3
BASELINE_BATCH    = 32        # larger batch — no GNN overhead

# Set True to skip CNN training and reuse saved checkpoints in CHECKPOINT_DIR
SKIP_CNN_TRAINING = False


def resolve_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = resolve_device()
print(f"Device: {DEVICE}")

Device: mps


In [4]:
# ============ FASTA and Label Loading (identical to v4.3) ============

def read_fasta(path):
    """Read FASTA file and return headers and sequences."""
    headers, sequences = [], []
    h, buf = None, []
    with open(path, 'r') as f:
        for line in f:
            if not line:
                continue
            if line[0] == '>':
                if h is not None:
                    sequences.append(''.join(buf).upper())
                    buf = []
                h = line[1:].strip()
                headers.append(h)
            else:
                buf.append(line.strip())
        if h is not None:
            sequences.append(''.join(buf).upper())
    return headers, sequences


def load_multiclass_labels(label_path, keep_classes=('DNA', 'LTR', 'LINE')):
    """
    Load labels for multi-class hierarchical classification.
    Returns label_dict (header->tag) and class_dict (header->class_id).
    """
    label_path = Path(label_path)
    class_to_id = {c: i for i, c in enumerate(keep_classes)}
    label_dict, class_dict = {}, {}
    superfamilies = Counter()
    skipped_classes = Counter()

    with label_path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            header = parts[0].lstrip('>')
            tag = parts[1]
            top_class = tag.split('/')[0]
            if top_class in class_to_id:
                label_dict[header] = tag
                class_dict[header] = class_to_id[top_class]
                superfamilies[tag] += 1
            else:
                skipped_classes[tag] += 1

    print(f"Loaded {len(label_dict)} sequences (filtered to {list(keep_classes)})")
    return label_dict, class_dict


def compute_class_weights(y_ids, n_classes, mode="inv_sqrt", eps=1e-6):
    """Compute class weights for imbalanced multi-class training."""
    counts = np.bincount(np.asarray(y_ids, dtype=np.int64), minlength=n_classes).astype(np.float64)
    if mode == "inv_sqrt":
        w = 1.0 / np.sqrt(counts + eps)
    elif mode == "inv":
        w = 1.0 / (counts + eps)
    else:
        w = np.ones(n_classes, dtype=np.float64)
    w = w / (w.mean() + 1e-12)
    return w.astype(np.float32)


print("Data utilities loaded.")

Data utilities loaded.


In [5]:
# ============ K-mer Feature Extraction (identical to v4.3) ============

# ASCII -> {0,1,2,3,4} for A,C,G,T,other
_ASCII_MAP = np.full(256, 4, dtype=np.uint8)
for _ch, _val in [("A", 0), ("C", 1), ("G", 2), ("T", 3),
                  ("a", 0), ("c", 1), ("g", 2), ("t", 3)]:
    _ASCII_MAP[ord(_ch)] = _val

_COMP = np.array([3, 2, 1, 0], dtype=np.uint8)  # A<->T, C<->G


def kmer_code_forward(arr4: np.ndarray) -> int:
    code = 0
    for v in arr4:
        code = (code << 2) | int(v)
    return code


def kmer_code_rc(arr4: np.ndarray) -> int:
    code = 0
    for v in arr4[::-1]:
        code = (code << 2) | int(_COMP[v])
    return code


def canonical_kmer_code(arr4: np.ndarray) -> int:
    c1 = kmer_code_forward(arr4)
    c2 = kmer_code_rc(arr4)
    return c1 if c1 < c2 else c2


def hash_u32(x: int, dim: int) -> int:
    z = (x * 0x9E3779B97F4A7C15) & 0xFFFFFFFFFFFFFFFF
    z ^= (z >> 33)
    z = (z * 0xC2B2AE3D27D4EB4F) & 0xFFFFFFFFFFFFFFFF
    z ^= (z >> 29)
    return int(z % dim)


@dataclass
class KmerWindowFeaturizer:
    """Extract k-mer frequency features from sliding windows."""
    k: int = 7
    dim: int = 2048
    window: int = 512
    stride: int = 256
    add_pos: bool = True
    l2_normalize: bool = True

    def featurize_sequence(self, seq: str) -> Tuple[np.ndarray, np.ndarray]:
        arr = _ASCII_MAP[np.frombuffer(seq.encode("ascii", "ignore"), dtype=np.uint8)]
        L = int(arr.size)

        if L == 0:
            X = np.zeros((1, self.dim + (1 if self.add_pos else 0)), dtype=np.float32)
            return X, np.array([0], dtype=np.int64)

        if L <= self.window:
            starts = np.array([0], dtype=np.int64)
        else:
            starts = np.arange(0, L - self.window + 1, self.stride, dtype=np.int64)
            if starts.size == 0:
                starts = np.array([0], dtype=np.int64)

        out_dim = self.dim + (1 if self.add_pos else 0)
        X = np.zeros((starts.size, out_dim), dtype=np.float32)

        for wi, st in enumerate(starts):
            en = min(st + self.window, L)
            sub = arr[st:en]
            counts = np.zeros(self.dim, dtype=np.float32)
            total = 0
            if sub.size >= self.k:
                for i in range(0, sub.size - self.k + 1):
                    kmer = sub[i:i + self.k]
                    if np.any(kmer == 4):
                        continue
                    code = canonical_kmer_code(kmer)
                    j = hash_u32(code, self.dim)
                    counts[j] += 1.0
                    total += 1
            if total > 0:
                counts /= float(total)
            if self.l2_normalize:
                nrm = np.linalg.norm(counts)
                if nrm > 0:
                    counts /= nrm
            if self.add_pos:
                center = (st + en) / 2.0
                pos = center / max(1.0, float(L))
                X[wi, :-1] = counts
                X[wi, -1] = pos
            else:
                X[wi, :] = counts

        return X, starts


print("KmerWindowFeaturizer loaded.")

KmerWindowFeaturizer loaded.


In [6]:
# ============ Load & Preprocess Dataset ============
# Mirrors v4.3 data pipeline for a consistent train/test split.

def _extract_genome_id(header: str) -> str:
    """Extract genome ID from VGP FASTA header (e.g. 'hAT_1-aAnoBae#DNA/hAT' -> 'aAnoBae')."""
    name_part = header.split('#')[0]
    return name_part.rsplit('-', 1)[-1]


def load_and_split_dataset():
    """
    Load the VGP dataset and return a dict with:
      sequences, class_labels, sf_labels, kmer_features,
      idx_train, idx_test, class_names, superfamily_names, n_classes, n_sf
    """
    class_to_id = {c: i for i, c in enumerate(KEEP_CLASSES)}

    print("=== Loading FASTA & labels ===")
    headers, sequences = read_fasta(FASTA_PATH)
    label_dict, class_dict = load_multiclass_labels(LABEL_PATH, keep_classes=KEEP_CLASSES)

    # Match headers; exclude benchmark genomes (same as v4.3)
    all_h, all_s, all_tags, all_toplevel = [], [], [], []
    n_excluded = 0
    for h, s in zip(headers, sequences):
        if h not in label_dict:
            continue
        if _extract_genome_id(h) in EXCLUDE_GENOMES:
            n_excluded += 1
            continue
        all_h.append(h)
        all_s.append(s)
        all_tags.append(label_dict[h])
        all_toplevel.append(class_dict[h])

    print(f"Matched {len(all_h)} sequences ({n_excluded} excluded from benchmark genomes)")
    del headers, sequences
    gc.collect()

    # Build superfamily mapping (min_count=100, same as v4.3)
    tag_counts = Counter(all_tags)
    keep_sfs = {t for t, c in tag_counts.items() if c >= MIN_CLASS_COUNT}
    superfamily_names = sorted(keep_sfs)
    sf_to_id = {t: i for i, t in enumerate(superfamily_names)}
    n_sf = len(superfamily_names)
    print(f"Superfamilies retained: {n_sf}")

    # Filter to sequences with valid superfamily
    fh, fs, ftags, ftop, fsf = [], [], [], [], []
    for h, s, tag, top in zip(all_h, all_s, all_tags, all_toplevel):
        if tag in sf_to_id:
            fh.append(h); fs.append(s); ftags.append(tag)
            ftop.append(top); fsf.append(sf_to_id[tag])
    all_h, all_s, all_tags = fh, fs, ftags
    all_toplevel = np.array(ftop, dtype=np.int64)
    all_sf       = np.array(fsf,  dtype=np.int64)
    del fh, fs, ftags, ftop, fsf
    gc.collect()

    # Subsample large superfamilies (max 3000 per SF, same as v4.3)
    np.random.seed(RANDOM_STATE)
    keep_idx = []
    for sf_name, sf_id in sf_to_id.items():
        idxs = np.where(all_sf == sf_id)[0]
        if len(idxs) > MAX_PER_SF:
            idxs = np.random.choice(idxs, MAX_PER_SF, replace=False)
        keep_idx.extend(idxs)
    keep_idx = sorted(keep_idx)
    all_h        = [all_h[i]        for i in keep_idx]
    all_s        = [all_s[i]        for i in keep_idx]
    all_tags     = [all_tags[i]     for i in keep_idx]
    all_toplevel = all_toplevel[keep_idx]
    all_sf       = all_sf[keep_idx]
    print(f"After subsampling (max {MAX_PER_SF}/SF): {len(all_h)} sequences")

    # Pre-compute k-mer features (needed for LR / RF baselines)
    print("=== Pre-computing k-mer features ===")
    featurizer = KmerWindowFeaturizer(
        k=KMER_K, dim=KMER_DIM, window=KMER_WINDOW,
        stride=KMER_STRIDE, add_pos=True, l2_normalize=True
    )
    kmer_features = []
    for seq in tqdm(all_s, desc="K-mer featurization"):
        X, _ = featurizer.featurize_sequence(seq)
        kmer_features.append(X)

    # Stratified train/test split — same parameters as v4.3
    strat_labels = np.array(all_tags)
    idx_train, idx_test = train_test_split(
        np.arange(len(all_h)), test_size=TEST_SIZE,
        stratify=strat_labels, random_state=RANDOM_STATE
    )
    print(f"Train: {len(idx_train)} | Test (held-out): {len(idx_test)}")

    return {
        'sequences':        all_s,
        'class_labels':     all_toplevel,
        'sf_labels':        all_sf,
        'kmer_features':    kmer_features,
        'idx_train':        idx_train,
        'idx_test':         idx_test,
        'class_names':      KEEP_CLASSES,
        'superfamily_names': superfamily_names,
        'sf_to_id':         sf_to_id,
        'n_classes':        len(KEEP_CLASSES),
        'n_sf':             n_sf,
    }


data = load_and_split_dataset()
print(f"\nDataset ready: {len(data['sequences'])} seqs | "
      f"{data['n_classes']} classes | {data['n_sf']} superfamilies")

=== Loading FASTA & labels ===
Loaded 50431 sequences (filtered to ['DNA', 'LTR', 'LINE'])
Matched 50403 sequences (28 excluded from benchmark genomes)
Superfamilies retained: 23
After subsampling (max 3000/SF): 30701 sequences
=== Pre-computing k-mer features ===


K-mer featurization:   0%|          | 0/30701 [00:00<?, ?it/s]

Train: 24560 | Test (held-out): 6141

Dataset ready: 30701 seqs | 3 classes | 23 superfamilies


## ML Baselines: Logistic Regression & Random Forest

Train on **mean-pooled k-mer window features** — the same feature space used by the k-mer GNN tower in v4.3, but without any sequence structure.

In [7]:
# ============ Prepare k-mer Feature Matrix ============

idx_tr = data['idx_train']
idx_te = data['idx_test']

print("Building mean-pooled k-mer feature matrix ...")
# Mean-pool window features -> (N, kmer_dim+1) flat vector per sequence
X_kmer = np.vstack([
    data['kmer_features'][i].mean(axis=0)
    for i in tqdm(range(len(data['sequences'])), desc="Mean-pooling")
])

X_train_km = X_kmer[idx_tr]
X_test_km  = X_kmer[idx_te]

y_class_train = data['class_labels'][idx_tr]
y_class_test  = data['class_labels'][idx_te]
y_sf_train    = data['sf_labels'][idx_tr]
y_sf_test     = data['sf_labels'][idx_te]

print(f"Feature matrix: {X_train_km.shape} train  |  {X_test_km.shape} test")

ml_results = {}  # populated below

Building mean-pooled k-mer feature matrix ...


Mean-pooling:   0%|          | 0/30701 [00:00<?, ?it/s]

Feature matrix: (24560, 2049) train  |  (6141, 2049) test


In [8]:
# ============ Logistic Regression ============

print("--- Logistic Regression: 3-class (DNA / LTR / LINE) ---")
t0 = time.time()
lr_cls = LogisticRegression(max_iter=500, C=1.0, random_state=RANDOM_STATE, n_jobs=-1)
lr_cls.fit(X_train_km, y_class_train)
lr_class_time = time.time() - t0
y_pred = lr_cls.predict(X_test_km)
lr_cls_f1  = f1_score(y_class_test, y_pred, average='macro')
lr_cls_acc = accuracy_score(y_class_test, y_pred)
print(f"  Train time: {lr_class_time:.1f}s  |  Macro-F1: {lr_cls_f1:.4f}  |  Acc: {lr_cls_acc:.4f}")
print(classification_report(y_class_test, y_pred, target_names=data['class_names']))

print("\n--- Logistic Regression: 23-class superfamily ---")
t0 = time.time()
lr_sf = LogisticRegression(max_iter=500, C=1.0, random_state=RANDOM_STATE, n_jobs=-1)
lr_sf.fit(X_train_km, y_sf_train)
lr_sf_time = time.time() - t0
y_pred_sf = lr_sf.predict(X_test_km)
lr_sf_f1  = f1_score(y_sf_test, y_pred_sf, average='macro')
lr_sf_acc = accuracy_score(y_sf_test, y_pred_sf)
print(f"  Train time: {lr_sf_time:.1f}s  |  Macro-F1: {lr_sf_f1:.4f}  |  Acc: {lr_sf_acc:.4f}")

ml_results['LR'] = {
    'class_f1':     lr_cls_f1,
    'class_acc':    lr_cls_acc,
    'sf_f1':        lr_sf_f1,
    'sf_acc':       lr_sf_acc,
    'train_time_s': lr_class_time + lr_sf_time,
    'model_size_mb': 0.0,
}
print("\nLogistic Regression results saved.")

--- Logistic Regression: 3-class (DNA / LTR / LINE) ---
  Train time: 8.0s  |  Macro-F1: 0.9401  |  Acc: 0.9396
              precision    recall  f1-score   support

         DNA       0.96      0.93      0.95      1697
         LTR       0.92      0.96      0.94      2506
        LINE       0.95      0.92      0.94      1938

    accuracy                           0.94      6141
   macro avg       0.94      0.94      0.94      6141
weighted avg       0.94      0.94      0.94      6141


--- Logistic Regression: 23-class superfamily ---
  Train time: 22.9s  |  Macro-F1: 0.6157  |  Acc: 0.7935

Logistic Regression results saved.


In [9]:
# ============ Random Forest ============

print("--- Random Forest: 3-class (DNA / LTR / LINE) ---")
t0 = time.time()
rf_cls = RandomForestClassifier(n_estimators=200, max_depth=None,
                                random_state=RANDOM_STATE, n_jobs=-1)
rf_cls.fit(X_train_km, y_class_train)
rf_class_time = time.time() - t0
y_pred = rf_cls.predict(X_test_km)
rf_cls_f1  = f1_score(y_class_test, y_pred, average='macro')
rf_cls_acc = accuracy_score(y_class_test, y_pred)
print(f"  Train time: {rf_class_time:.1f}s  |  Macro-F1: {rf_cls_f1:.4f}  |  Acc: {rf_cls_acc:.4f}")
print(classification_report(y_class_test, y_pred, target_names=data['class_names']))

print("\n--- Random Forest: 23-class superfamily ---")
t0 = time.time()
rf_sf = RandomForestClassifier(n_estimators=200, max_depth=None,
                               random_state=RANDOM_STATE, n_jobs=-1)
rf_sf.fit(X_train_km, y_sf_train)
rf_sf_time = time.time() - t0
y_pred_sf = rf_sf.predict(X_test_km)
rf_sf_f1  = f1_score(y_sf_test, y_pred_sf, average='macro')
rf_sf_acc = accuracy_score(y_sf_test, y_pred_sf)
print(f"  Train time: {rf_sf_time:.1f}s  |  Macro-F1: {rf_sf_f1:.4f}  |  Acc: {rf_sf_acc:.4f}")

ml_results['RF'] = {
    'class_f1':     rf_cls_f1,
    'class_acc':    rf_cls_acc,
    'sf_f1':        rf_sf_f1,
    'sf_acc':       rf_sf_acc,
    'train_time_s': rf_class_time + rf_sf_time,
    'model_size_mb': 0.0,
}
print("\nRandom Forest results saved.")

--- Random Forest: 3-class (DNA / LTR / LINE) ---
  Train time: 13.8s  |  Macro-F1: 0.9252  |  Acc: 0.9238
              precision    recall  f1-score   support

         DNA       0.98      0.93      0.95      1697
         LTR       0.87      0.98      0.92      2506
        LINE       0.96      0.84      0.90      1938

    accuracy                           0.92      6141
   macro avg       0.94      0.92      0.93      6141
weighted avg       0.93      0.92      0.92      6141


--- Random Forest: 23-class superfamily ---
  Train time: 13.3s  |  Macro-F1: 0.7725  |  Acc: 0.8199

Random Forest results saved.


## CNN Baselines

Two CNN-only architectures trained on raw sequences (one-hot encoded):
- **Standard CNN** — multi-scale motif convolutions + global average pool; **no** dilated context blocks  
- **Dilated CNN** — same motif convolutions + 4 dilated context blocks; **no** GNN tower or cross-modal fusion

In [10]:
# ============ CNN Baseline Dataset & Collate ============

# ASCII map for one-hot encoding
_ENCODE = np.zeros(256, dtype=np.uint8)
for _i, _c in enumerate('ACGTN'):
    _ENCODE[ord(_c)] = _i
    _ENCODE[ord(_c.lower())] = _i
_ENCODE[ord('N')] = 4
_ENCODE[ord('n')] = 4

_REV_COMP = torch.tensor([3, 2, 1, 0, 4], dtype=torch.long)


class SimpleDNADataset(Dataset):
    """One-hot encoded DNA sequences for CNN-only baselines (no GNN)."""

    def __init__(self, sequences, class_labels, sf_labels, fixed_length=FIXED_LENGTH):
        self.sequences    = sequences
        self.class_labels = np.asarray(class_labels, dtype=np.int64)
        self.sf_labels    = np.asarray(sf_labels,    dtype=np.int64)
        self.fixed_length = fixed_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        seq_len = len(seq)
        seq_idx = _ENCODE[np.frombuffer(seq.encode("ascii", "ignore"), dtype=np.uint8)]
        max_start = max(0, self.fixed_length - seq_len)
        start = np.random.randint(0, max_start + 1) if max_start > 0 else 0
        return (seq_idx, start, seq_len,
                int(self.class_labels[idx]), int(self.sf_labels[idx]))


def collate_cnn(batch, fixed_length=FIXED_LENGTH):
    seq_idxs, starts, lengths, y_cls, y_sf = zip(*batch)
    B = len(batch)
    X    = torch.zeros((B, 5, fixed_length), dtype=torch.float32)
    mask = torch.zeros((B, fixed_length),    dtype=torch.bool)
    for i, (si, st, sl) in enumerate(zip(seq_idxs, starts, lengths)):
        al = min(sl, fixed_length - st)
        if al > 0:
            idx = torch.from_numpy(si[:al].astype(np.int64))
            pos = torch.arange(al, dtype=torch.long) + st
            X[i, idx, pos] = 1.0
            mask[i, st:st + al] = (idx != 4)
    return (X, mask,
            torch.tensor(y_cls, dtype=torch.long),
            torch.tensor(y_sf,  dtype=torch.long))


print("SimpleDNADataset and collate_cnn loaded.")

SimpleDNADataset and collate_cnn loaded.


In [11]:
# ============ CNN Model Definitions ============

# ---- Shared building blocks (mirrors v4.3) ----

class ConvBlock(nn.Module):
    """Residual conv block with optional dilation."""
    def __init__(self, c_in, c_out, kernel_size=9, dilation=1, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, padding=pad, dilation=dilation)
        self.bn   = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1)

    def forward(self, x):
        y = self.drop(F.gelu(self.bn(self.conv(x))))
        return y + self.proj(x)


class MaskedMaxPool1d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride      = stride

    def forward(self, x, mask):
        if mask is not None:
            m = mask.unsqueeze(1).float()
            x = x * m + (~mask.unsqueeze(1)) * (-1e9)
        x_p = F.max_pool1d(x, self.kernel_size, self.stride)
        if mask is None:
            return x_p, None
        m_p = F.max_pool1d(mask.float().unsqueeze(1),
                           self.kernel_size, self.stride).squeeze(1) > 0
        return x_p, m_p


def masked_avg_pool(z, mask):
    if mask is None:
        return z.mean(-1)
    m = mask.unsqueeze(1).float()
    return (z * m).sum(-1) / m.sum(-1).clamp_min(1.0)


# ---- Baseline 1: Standard CNN (no dilated context blocks) ----

class StandardCNNClassifier(nn.Module):
    """
    Baseline CNN — motif convolutions + global average pool only.
    No dilated context blocks; no GNN or attention fusion.
    Represents a typical 'standard CNN' used as a comparison point.
    """
    def __init__(self, width=128, motif_kernels=(7, 15, 21),
                 dropout=0.15, n_classes=3, n_sf=23):
        super().__init__()
        self.motif_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(5, width, kernel_size=k, padding=k // 2, bias=True),
                nn.BatchNorm1d(width), nn.GELU(), nn.Dropout(dropout),
            ) for k in motif_kernels
        ])
        self.mix = nn.Sequential(
            nn.Conv1d(width * len(motif_kernels), width, 1),
            nn.BatchNorm1d(width), nn.GELU(), nn.Dropout(dropout),
        )
        self.class_head = nn.Sequential(
            nn.Linear(width, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, n_classes)
        )
        self.sf_head = nn.Sequential(
            nn.Linear(width, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, n_sf)
        )

    def forward(self, x, mask):
        feats = [conv(x) for conv in self.motif_convs]
        emb   = masked_avg_pool(self.mix(torch.cat(feats, dim=1)), mask)
        return self.class_head(emb), self.sf_head(emb)


# ---- Baseline 2: Dilated CNN (dilated context blocks, no GNN) ----

class DilatedCNNClassifier(nn.Module):
    """
    Dilated CNN tower with dual classification heads.
    Same CNN architecture as v4.3's CNNTower, but without the GNN tower
    or cross-modal attention fusion.
    """
    def __init__(self, width=128, motif_kernels=(7, 15, 21),
                 context_dilations=(1, 2, 4, 8), dropout=0.15,
                 n_classes=3, n_sf=23):
        super().__init__()
        self.motif_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(5, width, kernel_size=k, padding=k // 2, bias=True),
                nn.BatchNorm1d(width), nn.GELU(), nn.Dropout(dropout),
            ) for k in motif_kernels
        ])
        self.mix = nn.Sequential(
            nn.Conv1d(width * len(motif_kernels), width, 1),
            nn.BatchNorm1d(width), nn.GELU(), nn.Dropout(dropout),
        )
        self.context_blocks = nn.ModuleList([
            ConvBlock(width, width, kernel_size=9, dilation=d, dropout=dropout)
            for d in context_dilations
        ])
        self.pool = MaskedMaxPool1d(kernel_size=2, stride=2)
        self.class_head = nn.Sequential(
            nn.Linear(width, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, n_classes)
        )
        self.sf_head = nn.Sequential(
            nn.Linear(width, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, n_sf)
        )

    def forward(self, x, mask):
        feats = [conv(x) for conv in self.motif_convs]
        z, m  = self.mix(torch.cat(feats, dim=1)), mask
        for block in self.context_blocks:
            z = block(z)
            z, m = self.pool(z, m)
        emb = masked_avg_pool(z, m)
        return self.class_head(emb), self.sf_head(emb)


print("StandardCNNClassifier and DilatedCNNClassifier defined.")

StandardCNNClassifier and DilatedCNNClassifier defined.


In [12]:
# ============ CNN Baseline Training Loop ============

def train_cnn_baseline(
    model_name: str,
    model: nn.Module,
    train_seqs, test_seqs,
    train_cls, test_cls,
    train_sf,  test_sf,
    n_epochs:   int   = BASELINE_EPOCHS,
    lr:         float = BASELINE_LR,
    batch_size: int   = BASELINE_BATCH,
    device            = DEVICE,
    checkpoint_dir    = CHECKPOINT_DIR,
):
    """
    Train a CNN baseline and return a results dict.
    If SKIP_CNN_TRAINING=True and a checkpoint already exists, reload it.
    """
    ckpt_path = Path(checkpoint_dir) / f"{model_name.replace(' ', '_')}_best.pt"

    if SKIP_CNN_TRAINING and ckpt_path.exists():
        print(f"[{model_name}] Loading cached checkpoint: {ckpt_path}")
        saved = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        return saved['results']

    train_ds = SimpleDNADataset(train_seqs, train_cls, train_sf)
    test_ds  = SimpleDNADataset(test_seqs,  test_cls,  test_sf)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          collate_fn=collate_cnn, num_workers=0)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size * 2, shuffle=False,
                          collate_fn=collate_cnn, num_workers=0)

    model     = model.to(device)
    n_params  = sum(p.numel() for p in model.parameters())
    model_mb  = n_params * 4 / 1e6

    # Inverse-sqrt class weights (same strategy as v4.3)
    cls_w = torch.tensor(
        compute_class_weights(train_cls, data['n_classes'], mode='inv_sqrt'),
        dtype=torch.float32
    ).to(device)
    sf_w = torch.tensor(
        compute_class_weights(train_sf, data['n_sf'], mode='inv_sqrt'),
        dtype=torch.float32
    ).to(device)

    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = CosineAnnealingLR(optimizer, T_max=n_epochs)
    crit_cls   = nn.CrossEntropyLoss(weight=cls_w)
    crit_sf    = nn.CrossEntropyLoss(weight=sf_w)

    best_combined = -1.0
    best_state    = None
    best_results  = None
    t_start       = time.time()

    for epoch in range(1, n_epochs + 1):
        # ---- Train ----
        model.train()
        ep_loss = 0.0
        for X, mask, y_cls, y_sf in train_dl:
            X, mask, y_cls, y_sf = (X.to(device), mask.to(device),
                                     y_cls.to(device), y_sf.to(device))
            optimizer.zero_grad()
            logits_cls, logits_sf = model(X, mask)
            loss = crit_cls(logits_cls, y_cls) + crit_sf(logits_sf, y_sf)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
        scheduler.step()

        # ---- Evaluate on held-out test set ----
        model.eval()
        cls_preds, cls_trues = [], []
        sf_preds,  sf_trues  = [], []
        with torch.no_grad():
            for X, mask, y_cls, y_sf in test_dl:
                X, mask = X.to(device), mask.to(device)
                logits_cls, logits_sf = model(X, mask)
                cls_preds.extend(logits_cls.argmax(1).cpu().tolist())
                cls_trues.extend(y_cls.tolist())
                sf_preds.extend(logits_sf.argmax(1).cpu().tolist())
                sf_trues.extend(y_sf.tolist())

        cls_f1 = f1_score(cls_trues, cls_preds, average='macro')
        sf_f1  = f1_score(sf_trues,  sf_preds,  average='macro')
        combined = 0.5 * cls_f1 + 0.5 * sf_f1

        if epoch % 5 == 0 or epoch == n_epochs:
            print(f"  [{model_name}] Epoch {epoch:2d}/{n_epochs} | "
                  f"loss={ep_loss/len(train_dl):.4f} | "
                  f"cls_F1={cls_f1:.4f} | sf_F1={sf_f1:.4f}")

        if combined > best_combined:
            best_combined = combined
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_results  = {
                'class_f1':     cls_f1,
                'class_acc':    accuracy_score(cls_trues, cls_preds),
                'sf_f1':        sf_f1,
                'sf_acc':       accuracy_score(sf_trues,  sf_preds),
                'train_time_s': time.time() - t_start,
                'model_size_mb': model_mb,
                'n_params':     n_params,
                '_cls_preds':   cls_preds,
                '_cls_trues':   cls_trues,
            }

    best_results['train_time_s'] = time.time() - t_start

    torch.save({'model': best_state, 'results': best_results}, ckpt_path)
    print(f"\n[{model_name}] BEST — cls_F1={best_results['class_f1']:.4f} | "
          f"sf_F1={best_results['sf_f1']:.4f} | "
          f"time={best_results['train_time_s']:.0f}s | {model_mb:.1f} MB")
    print(f"\n{model_name} — 3-class classification report:")
    print(classification_report(
        best_results['_cls_trues'], best_results['_cls_preds'],
        target_names=data['class_names']
    ))
    return best_results


cnn_results = {}  # populated in next two cells
print("train_cnn_baseline() ready.")

train_cnn_baseline() ready.


In [13]:
# ============ Train Standard CNN (no dilation) ============

print("=== Training Standard CNN (no dilated blocks) ===")

std_cnn = StandardCNNClassifier(
    width=CNN_WIDTH, motif_kernels=MOTIF_KERNELS, dropout=DROPOUT,
    n_classes=data['n_classes'], n_sf=data['n_sf']
)
print(f"Parameters: {sum(p.numel() for p in std_cnn.parameters()):,}")

cnn_results['Standard CNN'] = train_cnn_baseline(
    model_name  = "Standard CNN",
    model       = std_cnn,
    train_seqs  = [data['sequences'][i] for i in idx_tr],
    test_seqs   = [data['sequences'][i] for i in idx_te],
    train_cls   = data['class_labels'][idx_tr],
    test_cls    = data['class_labels'][idx_te],
    train_sf    = data['sf_labels'][idx_tr],
    test_sf     = data['sf_labels'][idx_te],
)

=== Training Standard CNN (no dilated blocks) ===
Parameters: 106,138
  [Standard CNN] Epoch  5/20 | loss=1.5986 | cls_F1=0.9218 | sf_F1=0.5445
  [Standard CNN] Epoch 10/20 | loss=1.1295 | cls_F1=0.8477 | sf_F1=0.5919
  [Standard CNN] Epoch 15/20 | loss=0.9065 | cls_F1=0.9613 | sf_F1=0.7096
  [Standard CNN] Epoch 20/20 | loss=0.8046 | cls_F1=0.9697 | sf_F1=0.7244

[Standard CNN] BEST — cls_F1=0.9697 | sf_F1=0.7244 | time=13688s | 0.4 MB

Standard CNN — 3-class classification report:
              precision    recall  f1-score   support

         DNA       0.98      0.96      0.97      1697
         LTR       0.96      0.98      0.97      2506
        LINE       0.97      0.96      0.97      1938

    accuracy                           0.97      6141
   macro avg       0.97      0.97      0.97      6141
weighted avg       0.97      0.97      0.97      6141



In [14]:
# ============ Train Dilated CNN (dilated blocks, no GNN) ============

print("=== Training Dilated CNN (with dilated context, no GNN) ===")

dil_cnn = DilatedCNNClassifier(
    width=CNN_WIDTH, motif_kernels=MOTIF_KERNELS,
    context_dilations=CONTEXT_DILATIONS, dropout=DROPOUT,
    n_classes=data['n_classes'], n_sf=data['n_sf']
)
print(f"Parameters: {sum(p.numel() for p in dil_cnn.parameters()):,}")

cnn_results['Dilated CNN'] = train_cnn_baseline(
    model_name  = "Dilated CNN",
    model       = dil_cnn,
    train_seqs  = [data['sequences'][i] for i in idx_tr],
    test_seqs   = [data['sequences'][i] for i in idx_te],
    train_cls   = data['class_labels'][idx_tr],
    test_cls    = data['class_labels'][idx_te],
    train_sf    = data['sf_labels'][idx_tr],
    test_sf     = data['sf_labels'][idx_te],
)

=== Training Dilated CNN (with dilated context, no GNN) ===
Parameters: 697,498
  [Dilated CNN] Epoch  5/20 | loss=1.7575 | cls_F1=0.9568 | sf_F1=0.3526
  [Dilated CNN] Epoch 10/20 | loss=1.3218 | cls_F1=0.9664 | sf_F1=0.5768
  [Dilated CNN] Epoch 15/20 | loss=1.0091 | cls_F1=0.9791 | sf_F1=0.5245
  [Dilated CNN] Epoch 20/20 | loss=0.8642 | cls_F1=0.9829 | sf_F1=0.5732

[Dilated CNN] BEST — cls_F1=0.9864 | sf_F1=0.6116 | time=9731s | 2.8 MB

Dilated CNN — 3-class classification report:
              precision    recall  f1-score   support

         DNA       0.99      0.98      0.99      1697
         LTR       0.98      0.99      0.99      2506
        LINE       0.99      0.98      0.99      1938

    accuracy                           0.99      6141
   macro avg       0.99      0.99      0.99      6141
weighted avg       0.99      0.99      0.99      6141



## Advanced Model Results

Load saved test-set predictions from the v4.3 Hybrid (CNN+GNN) model.

In [15]:
# ============ Load v4.3 Hybrid Results ============

v43_raw = torch.load(RESULTS_V43, map_location='cpu', weights_only=False)

v43_cls_f1  = f1_score(v43_raw['test_class_true'], v43_raw['test_class_pred'], average='macro')
v43_sf_f1   = f1_score(v43_raw['test_sf_true'],    v43_raw['test_sf_pred'],    average='macro')
v43_cls_acc = accuracy_score(v43_raw['test_class_true'], v43_raw['test_class_pred'])
v43_sf_acc  = accuracy_score(v43_raw['test_sf_true'],    v43_raw['test_sf_pred'])

print(f"v4.3 Hybrid (CNN+GNN)")
print(f"  3-class  — Macro-F1: {v43_cls_f1:.4f}  |  Acc: {v43_cls_acc:.4f}")
print(f"  23-class — Macro-F1: {v43_sf_f1:.4f}  |  Acc: {v43_sf_acc:.4f}")
print(f"  Best epoch: {v43_raw['best_epoch']}")

print("\nv4.3 — 3-class classification report:")
print(classification_report(
    v43_raw['test_class_true'], v43_raw['test_class_pred'],
    target_names=v43_raw['class_names']
))

v4.3 Hybrid (CNN+GNN)
  3-class  — Macro-F1: 0.9814  |  Acc: 0.9818
  23-class — Macro-F1: 0.8389  |  Acc: 0.8927
  Best epoch: 40

v4.3 — 3-class classification report:
              precision    recall  f1-score   support

         DNA       0.98      0.97      0.98      1697
         LTR       0.98      0.98      0.98      2506
        LINE       0.98      0.99      0.98      1938

    accuracy                           0.98      6141
   macro avg       0.98      0.98      0.98      6141
weighted avg       0.98      0.98      0.98      6141



## Compile All Results & Visualize

Aggregate metrics from all models and produce the comparison bar chart.

In [16]:
# ============ Compile Results ============

all_results = {}

# ML baselines (trained above)
for name, r in ml_results.items():
    all_results[name] = r

# CNN baselines (trained above)
for name, r in cnn_results.items():
    # Strip internal debug keys before serialising
    all_results[name] = {k: v for k, v in r.items() if not k.startswith('_')}

# v4.3 full hybrid — loaded from saved predictions
# Training time: ~40 epochs × ~60s/epoch on MPS (approximate; update if you timed it)
all_results['v4.3 Hybrid\n(CNN+GNN)'] = {
    'class_f1':      v43_cls_f1,
    'class_acc':     v43_cls_acc,
    'sf_f1':         v43_sf_f1,
    'sf_acc':        v43_sf_acc,
    'train_time_s':  40 * 60,   # ~40 min; adjust if you have a precise measurement
    'model_size_mb': 12.0,      # CNN + GNN towers + fusion heads (approximate)
}

# ---- Summary table ----
print(f"\n{'Model':<30} {'3-cls F1':>9} {'SF F1':>9} {'Train Time':>12} {'Size MB':>9}")
print("-" * 75)
for name, r in all_results.items():
    t   = r.get('train_time_s', 0)
    t_s = f"{t:.0f}s" if t < 120 else f"{t/60:.1f}m"
    print(f"{name.replace(chr(10),' '):<30} {r['class_f1']:>9.4f} {r['sf_f1']:>9.4f} "
          f"{t_s:>12} {r.get('model_size_mb',0):>9.1f}")

# ---- Persist to JSON ----
results_json = {
    k.replace('\n', ' '): {kk: (float(vv) if isinstance(vv, (float, np.floating, int)) else vv)
                            for kk, vv in v.items()}
    for k, v in all_results.items()
}
with open(CHECKPOINT_DIR / "baseline_results.json", 'w') as f:
    _json.dump(results_json, f, indent=2)
print("\nResults saved → baseline_results.json")


Model                           3-cls F1     SF F1   Train Time   Size MB
---------------------------------------------------------------------------
LR                                0.9401    0.6157          31s       0.0
RF                                0.9252    0.7725          27s       0.0
Standard CNN                      0.9697    0.7244       228.1m       0.4
Dilated CNN                       0.9864    0.6116       162.2m       2.8
v4.3 Hybrid (CNN+GNN)             0.9814    0.8389        40.0m      12.0

Results saved → baseline_results.json


In [17]:
# ============ Baseline Comparison Bar Chart ============

MODEL_ORDER = [
    'LR', 'RF', 'Standard CNN', 'Dilated CNN', 'v4.3 Hybrid\n(CNN+GNN)'
]
MODEL_LABELS = {
    'LR':                    'Log.\nRegression',
    'RF':                    'Random\nForest',
    'Standard CNN':          'Standard\nCNN',
    'Dilated CNN':           'Dilated\nCNN',
    'v4.3 Hybrid\n(CNN+GNN)': 'v4.3 Hybrid\n(CNN+GNN)',
}
PALETTE = {
    'LR':                    '#adb5bd',
    'RF':                    '#868e96',
    'Standard CNN':          '#74c0fc',
    'Dilated CNN':           '#339af0',
    'v4.3 Hybrid\n(CNN+GNN)': '#1864ab',
}

# ---- build ordered lists ----
models  = MODEL_ORDER
xlabels = [MODEL_LABELS[m] for m in models]
colors  = [PALETTE[m]      for m in models]
x       = np.arange(len(models))
bar_w   = 0.6

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(
    'Model Comparison — TE Transposon Classification\n'
    '(same train/test split; test set held-out from all training)',
    fontsize=12, fontweight='bold', y=1.02
)

# helper: annotate bars
def annotate(ax, bars, vals, fmt='.3f'):
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                val + 0.012, f'{val:{fmt}}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

# ---- Panel 1: 3-class macro-F1 ----
ax = axes[0]
vals = [all_results[m]['class_f1'] for m in models]
bars = ax.bar(x, vals, width=bar_w, color=colors, edgecolor='white', linewidth=0.8, zorder=3)
ax.axhline(all_results['v4.3 Hybrid\n(CNN+GNN)']['class_f1'],
           color='#1864ab', ls='--', lw=1.2, alpha=0.45, zorder=2)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Macro-F1', fontsize=11)
ax.set_title('3-Class (DNA / LTR / LINE)', fontsize=11, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=9)
ax.yaxis.grid(True, alpha=0.35, zorder=0); ax.set_axisbelow(True)
annotate(ax, bars, vals)

# ---- Panel 2: 23-class superfamily macro-F1 ----
ax = axes[1]
vals = [all_results[m]['sf_f1'] for m in models]
bars = ax.bar(x, vals, width=bar_w, color=colors, edgecolor='white', linewidth=0.8, zorder=3)
ax.axhline(all_results['v4.3 Hybrid\n(CNN+GNN)']['sf_f1'],
           color='#1864ab', ls='--', lw=1.2, alpha=0.45, zorder=2)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Macro-F1', fontsize=11)
ax.set_title('23-Class Superfamily', fontsize=11, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=9)
ax.yaxis.grid(True, alpha=0.35, zorder=0); ax.set_axisbelow(True)
annotate(ax, bars, vals)

# ---- Panel 3: Training time (log scale, minutes) ----
ax = axes[2]
vals_min = [max(all_results[m].get('train_time_s', 0.1) / 60, 0.005) for m in models]
bars = ax.bar(x, vals_min, width=bar_w, color=colors, edgecolor='white', linewidth=0.8, zorder=3)
ax.set_yscale('log')
ax.set_ylabel('Training time (minutes, log scale)', fontsize=11)
ax.set_title('Computational Cost', fontsize=11, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=9)
ax.yaxis.grid(True, alpha=0.35, zorder=0); ax.set_axisbelow(True)
for bar, val in zip(bars, vals_min):
    label = f'{val*60:.0f}s' if val < 1 else f'{val:.1f}m'
    ax.text(bar.get_x() + bar.get_width() / 2, val * 1.4,
            label, ha='center', va='bottom', fontsize=8, fontweight='bold')

# ---- Legend ----
legend_handles = [
    mpatches.Patch(color='#adb5bd', label='ML (k-mer features)'),
    mpatches.Patch(color='#74c0fc', label='CNN baseline'),
    mpatches.Patch(color='#1864ab', label='Hybrid CNN+GNN (v4.3)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           fontsize=10, bbox_to_anchor=(0.5, -0.07),
           frameon=True, framealpha=0.9)

plt.tight_layout()

# ---- Save ----
Path(FIGURE_PATH).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
print(f"Figure saved → {FIGURE_PATH}")
plt.show()

Figure saved → ../../thesis/figures/baseline_comparison.png


/var/folders/9v/zbs2wngj1392rlfhxv3090wm0000gn/T/ipykernel_94089/1862522789.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
